# Zero-Train Optimization & Maintenance (ZTOM) of a Healthcare Chatbot

**Medical Chatbot (Llama 3.2 + LoRA adapters)**

## Background

The Medical Chatbot is a large language model (LLM) based on Llama3.2 with LoRA adapters that was trained on online medical forums to provide medical advice from healthcare-related queries. The model processes medical questions and provides appropriate responses based on patterns learned from medical forum discussions.

**Dataset:** https://huggingface.co/datasets/lextale/FirstAidInstructionsDataset/viewer/default/icliniqDataset

## Scenario

After initial training, the model may benefit from optimization to improve its response quality and accuracy. Traditional approaches would require retraining the model with additional data, which is time-consuming and computationally expensive. ZTOM provides an alternative solution that optimizes the model's performance using only a small validation set and semantic similarity metrics to guide the optimization process.

## Use Case

This demonstration is applicable when:
- A model is already trained and deployed, but performance improvements are desired
- Retraining is not feasible due to time, computational, or data constraints
- You have access to only a small validation set for optimization
- You want to improve response quality using semantic similarity as a guide

## Summary

This notebook demonstrates how the Zero-Train Optimization & Maintenance (ZTOM) improves the Medical Chatbot's response quality without requiring any retraining. Using semantic similarity to guide optimization, ZTOM reduced the model's error rate by approximately 1.37%, demonstrating measurable improvements in response quality with minimal data requirements.
  
## ZTOM Result Outputs

  | Metric                       | Value                          |
  |------------------------------|-------------------------------:|
  | Non-Optimized Model Error    | 0.387                          |
  | Optimized Model Error        | 0.350                          |
  | Improvement                  | ~1.37%                         |
  | Scaling Factors              | [-0.089,-0.173,-0.175,-0.199]  |
  | Number of Inferences         | 60                             |

### Install dependencies and utility functions: This cell should be run once.

In [1]:
%%capture

%pip install matplotlib numpy pandas requests tqdm ipywidgets sentence-transformers
%pip install --ignore-installed 'git+https://github.com/Authentrics-ai/authentrics-client.git@v2.4.1'

from pathlib import Path

import os
import json
import authentrics_client as authrx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import random
from datetime import datetime
import time
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from threading import Thread
import contextlib
from sentence_transformers import SentenceTransformer

# Utility functions for loading indicators
@contextlib.contextmanager
def loading_spinner(message="Processing..."):
    """Context manager for showing a loading spinner"""
    spinner_widget = widgets.HTML(value=f"""
    <div style="display: flex; align-items: center; gap: 10px;">
        <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
        <span style="font-size: 14px; color: #555;">{message}</span>
    </div>
    <style>
    @keyframes spin {{
        0% {{ transform: rotate(0deg); }}
        100% {{ transform: rotate(360deg); }}
    }}
    </style>
    """)
    display(spinner_widget)
    try:
        yield
    finally:
        spinner_widget.close()

def show_progress_bar(total, description="Progress"):
    """Create and return a progress bar"""
    return tqdm(total=total, desc=description, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

def wait_for_analytics(project_id, min_wait_time=30, max_wait_time=300, check_interval=10):
    """
    Wait for ALL analytics to be complete before continuing.
    This prevents NaN values from appearing in the summary table.
    """
    import time

    print("🔄 Waiting for analytics to process...")
    countdown_widget = widgets.HTML()
    display(countdown_widget)

    # Minimum wait with countdown
    for remaining in range(min_wait_time, 0, -1):
        countdown_widget.value = f"""
        <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
            <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
            <span style="font-size: 14px; color: #555;">Minimum wait period: {remaining} seconds remaining...</span>
        </div>
        <style>
        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}
        </style>
        """
        time.sleep(1)

    # Poll until ALL analytics are complete
    start_polling = time.time()
    attempt = 1

    while (time.time() - start_polling) < (max_wait_time - min_wait_time):
        try:
            project = client.project.get_project_by_id(project_id)
            files = project["fileList"][1:]  # Skip base model

            # Check that ALL files have both weight and bias contributions
            files_missing_analytics = []
            for f in files:
                if (f.get("totalWeightContribution") is None or
                    f.get("totalBiasContribution") is None):
                    files_missing_analytics.append(f["fileName"])

            # Only continue if ALL analytics are complete
            if not files_missing_analytics:
                countdown_widget.value = """
                <div style="color: green; font-weight: bold; margin: 10px 0;">
                    ✅ All analytics are complete!
                </div>
                """
                return True

            # Show progress with specific missing files
            total_files = len(files)
            completed_files = total_files - len(files_missing_analytics)
            progress_pct = (completed_files / total_files * 100) if total_files > 0 else 0

            missing_display = ', '.join(files_missing_analytics[:3])
            if len(files_missing_analytics) > 3:
                missing_display += f" and {len(files_missing_analytics) - 3} more"

            countdown_widget.value = f"""
            <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
                <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #orange; border-radius: 50%; animation: spin 1s linear infinite;"></div>
                <span style="font-size: 14px; color: #555;">
                    Attempt {attempt}: {completed_files}/{total_files} files complete ({progress_pct:.1f}%)
                    <br/>⏳ Waiting for: {missing_display}
                </span>
            </div>
            <style>
            @keyframes spin {{
                0% {{ transform: rotate(0deg); }}
                100% {{ transform: rotate(360deg); }}
            }}
            </style>
            """
            attempt += 1
            time.sleep(check_interval)

        except Exception as e:
            print(f"⚠️ Error checking analytics status: {e}")
            time.sleep(check_interval)

    # Timeout reached - show final status
    try:
        project = client.project.get_project_by_id(project_id)
        files = project["fileList"][1:]
        files_missing_analytics = [f["fileName"] for f in files
                                 if (f.get("totalWeightContribution") is None or
                                     f.get("totalBiasContribution") is None)]

        if files_missing_analytics:
            countdown_widget.value = f"""
            <div style="color: red; font-weight: bold; margin: 10px 0;">
                ⚠️ Timeout: Analytics incomplete for {len(files_missing_analytics)} files.
                <br/>NaN values will appear for: {', '.join(files_missing_analytics[:5])}
                <br/>Consider increasing wait time or contacting support.
            </div>
            """
            return False
        else:
            countdown_widget.value = """
            <div style="color: green; font-weight: bold; margin: 10px 0;">
                ✅ All analytics completed just in time!
            </div>
            """
            return True
    except Exception as e:
        countdown_widget.value = f"""
        <div style="color: red; font-weight: bold; margin: 10px 0;">
            ❌ Error checking final status: {e}
        </div>
        """
        return False

def inference(
    client: authrx.AuthentricsClient,
    optimized_model_path: str,
    prompt: list[str] | Path,
    batch_size: int = 1,
):
    base_model_path = "demo-models/Medical-Chatbot/iteration_24.tar"

    # Single stimulus
    if isinstance(prompt, Path):
        optimized_result = client.dynamic.direct_inference(
            model_path=optimized_model_path,
            stimulus_path=prompt,
            format="HF_TEXT_GENERATION",
            inference_config={"max_new_tokens": 50},
        )
        base_result = client.dynamic.direct_inference(
            model_path=base_model_path,
            stimulus_path=prompt,
            format="HF_TEXT_GENERATION",
            inference_config={"max_new_tokens": 50},
        )
        return {
            "prompt": prompt,
            "model_responses": [
                base_result["model_responses"][0],
                optimized_result["model_responses"][0],
            ],
        }

    # Multiple stimuli
    else:
        optimized_response = client.dynamic.batch_direct_inference(
            model_path=optimized_model_path,
            stimulus_paths=prompt,
            format="HF_TEXT_GENERATION",
            batch_size=batch_size,
            inference_config={"max_new_tokens": 50},
        )
        base_response = client.dynamic.batch_direct_inference(
            model_path=base_model_path,
            stimulus_paths=prompt,
            format="HF_TEXT_GENERATION",
            batch_size=batch_size,
            inference_config={"max_new_tokens": 50},
        )

        return {
            "prompt": prompt,
            "model_responses": [
                base_response["model_responses"][0],
                optimized_response["model_responses"][0],
            ],
        }


def display_static_analysis_results(result: dict):
    print(f"Weight summary score: {result['weight_summary_score']}")
    pd.options.display.max_rows = None


    methods = {"min": np.amin, "mean": np.mean, "std": np.std, "max": np.amax}
    awd = {"name": [], "min": [], "mean": [], "std": [], "max": []}

    for name, values in response["result"]["absolute_weight_difference"].items():
        awd["name"].append(name)
        for method_name, method in methods.items():
            awd[method_name].append(method(values))

    df = pd.DataFrame(awd)
    display(df)


def get_ZTOM_results(project_id, request_id, initial_wait=15 * 60, max_attempts=30):
    """
    Polls the getAnalysisResults endpoint until a result is available or timeout is reached.
    Waits 15 minutes initially, then polls every 60 seconds.
    Returns the result dictionary if successful, raises an Exception otherwise.
    """
    poll_interval = 60  # Poll every 60 seconds
    # initial_wait = 15 * 60  # Wait 15 minutes (900 seconds) before starting to poll
    # max_attempts = 30  # Maximum number of polling attempts after initial wait

    # Wait 15 minutes before starting to poll with countdown
    print(f"Waiting {initial_wait // 60} minutes before starting to poll...")
    remaining_seconds = initial_wait
    while remaining_seconds > 0:
        minutes = remaining_seconds // 60
        seconds = remaining_seconds % 60
        print(f"\rTime remaining: {minutes:02d}:{seconds:02d}", end="", flush=True)
        time.sleep(1)
        remaining_seconds -= 1
    print("\nStarting to poll for results...")

    # Poll every 60 seconds
    for attempt in range(max_attempts):
        try:
            client.result.downloadAnalysisResultArtifact(project_id, request_id, "./result.json")
        except Exception as e:
            # If we get a 404 or other error, the result isn't ready yet - continue polling
            error_msg = str(e)
            if "404" in error_msg or "Not Found" in error_msg:
                print(f"Attempt {attempt + 1}/{max_attempts}: Result not found (404), waiting {poll_interval} seconds...")
            else:
                print(f"Attempt {attempt + 1}/{max_attempts}: Error ({error_msg[:50]}), waiting {poll_interval} seconds...")
            time.sleep(poll_interval)
            continue

        # Successfully downloaded - check if result is available
        try:
            with open("./result.json", "r") as f:
                response = json.load(f)

            if "trained_model_error" in response and response["trained_model_error"] is not None:
                return response
            else:
                # Result file exists but doesn't have the expected data yet
                print(f"Attempt {attempt + 1}/{max_attempts}: Result file exists but not ready yet, waiting {poll_interval} seconds...")
                time.sleep(poll_interval)
                continue
        except (FileNotFoundError, json.JSONDecodeError) as e:
            # File doesn't exist or is invalid JSON - continue polling
            print(f"Attempt {attempt + 1}/{max_attempts}: Error reading result file, waiting {poll_interval} seconds...")
            time.sleep(poll_interval)
            continue

    # If we get here, we've exhausted all attempts
    raise TimeoutError(f"Polling timed out after {max_attempts} attempts. Results may still be processing.")

### Contact Authentrics for the URL and user credentials info@authentrics.ai

In [2]:

DEMO_SERVER_URL = input("Enter your Authentrics URL: ")
PROJECT_NAME = "Healthcare Chatbot ZTOM"
pd.options.display.precision = 3
pd.options.display.chop_threshold = None

### Establish a session with authentrics
- Rerun this cell if your session expires

In [3]:
client = authrx.AuthentricsClient(DEMO_SERVER_URL)
client.auth.login()
print("✅ Session established successfully!")

✅ Session established successfully!


### Generate a project in Authentrics to track checkpoints, and perform asynchronous analytics
1. Creates a project for checkpoint management.
2. **Option A**: Point to existing checkpoint storage (no data movement required).
3. **Option B**: Upload checkpoints directly through our client (automated storage and versioning).

In [4]:
projects = client.project.get_projects()
if projects is not None:
    project = next((p for p in projects if p["name"].startswith(PROJECT_NAME)), None)
    if project:
        client.project.delete_project(project["id"], hard_delete=True)

PROJECT_NAME = PROJECT_NAME + " " + str(int(datetime.now().timestamp()))
project = client.project.create_project(
    PROJECT_NAME,
    "A smaller LLM specializing in medical advice",
    authrx.FileType.HF_TEXT,
)
project_id = project["id"]
print(f"✅ Project created with id: {project['id']}")

✅ Project created with id: 6970ddcafb2cee375a6181e1


### Restore all checkpoint files, stimulus files, and expected output file in bucket to a stable version

In [5]:
with loading_spinner("Please wait while we restore the requested files..."):
    client.get('/transfer/MedChat')

### Point authentrics file registry to the checkpoints using the api

In [6]:
print("Adding checkpoints to project...")
with show_progress_bar(5, "Adding checkpoints") as pbar:
    for i in range(20, 25):
        client.checkpoint.add_external_checkpoint(
            project_id,
            f"demo-models/Medical-Chatbot/iteration_{i}.tar",
            authrx.FileType.HF_TEXT,
            file_name=f"iteration_from_dataset_{i}.tar",
            tag=f"v{i}",
        )
        pbar.update(1)

project = client.project.get_project_by_name(PROJECT_NAME)
print(f"Project name: {project['name']}")
for file in project["fileList"]:
    print(f"File name: {file['fileName']}")

Adding checkpoints to project...


Project name: Healthcare Chatbot ZTOM 1769202747
File name: iteration_from_dataset_20.tar
File name: iteration_from_dataset_21.tar
File name: iteration_from_dataset_22.tar
File name: iteration_from_dataset_23.tar
File name: iteration_from_dataset_24.tar


Wait for Analytics to be ready before proceeding.

In [7]:
print("\n" + "="*60)
print("📊 ANALYTICS PROCESSING")
print("="*60)
print("Authentrics is now computing analytics in the background.")
print("This includes weight contributions and bias scores for each checkpoint.")

analytics_ready = wait_for_analytics(project_id, min_wait_time=10, max_wait_time=300)
if not analytics_ready:
    print("\n❌ Analytics are not complete.")
    print("Please wait a moment longer and then continue. Please contact support if the issue persists.")

## Example Prompt and Expected Output

Prompt: Hello sir,  My son has sinusitis and also nasal polyps, we got to know this recently. He suffers from a nose block and breathing issues. So when we visited a PCP (primary care physician), he suggested using nasal spray. So my doubt is, how can nasal spray efficiently relieve symptoms of sinusitis and nasal polyps, and what mechanisms make it useful for people suffering from these conditions? Could you also elaborate on how they work to reduce inflammation and congestion, and any possible negative effects or things to watch out for when using nasal sprays to treat sinusitis and nasal polyps?

Expected Output: You now have exclusive access to expert medical opinion.  Before I go to answer your questions, you need to understand the physiological and pathology of the problem in concern. Having understood that the treatment and mechanism of action of medications become much easier to understand.  The nasal cavity ...  Please do not hesitate to reach out if you have any further questions or concerns.  Thank you.

## Zero-Train Optimization & Maintenance (ZTOM) Results

Authentrics.ai provides a ZTOM service that allows you to optimize a model's performance without additional training.

This will increase or decrease the effects of each training iteration on the model to obtain the best possible performance. In this case, the performance is measured by the semantic similarity between the model's response and the expected output.

The scaling factor limit is the maximum fraction of change that any one training iteration can undergo. In this case, each training iteration can only change by 20% of its original effect.

In [8]:
stimulus_paths = [f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)]
expected_output_path = "demo-models/Medical-Chatbot/stimuli/expected_output.txt"

res = client.dynamic.zero_train_optimizer(
    project_id=project["id"],
    scaling_factor_limit=0.2,
    stimulus_paths=stimulus_paths,
    batch_size=10,
    expected_output_path=expected_output_path,
    inference_config={"max_new_tokens": 100},
    detailed_output="STORE_AND_RETURN"
)

request_id = res["requestId"]
print(request_id)

84546986-5143-4041-b61d-810e5f8c68a1
CPU times: user 2.28 ms, sys: 1.44 ms, total: 3.72 ms
Wall time: 373 ms


Wait for the result to complete processing. This will take around 15 minutes.

In [9]:
result = get_ZTOM_results(project_id, request_id, initial_wait=15 * 60, max_attempts=30)
print(result)

Waiting 15 minutes before starting to poll...

Starting to poll for results...
{'trained_model_error': 0.3869526982307434, 'optimized_model_error': 0.3503949046134949, 'optimized_scaling_factors': [-0.08874866916124041, -0.17300100314109593, -0.1752716729481579, -0.19940504499949116], 'number_of_inferences': 60 }


### Result details

`base_model_error` is the semantic similarity between the trained model's response and the expected output.

In [10]:
result["trained_model_error"]

0.3869526982307434

`optimized_model_error` is the semantic similarity between the optimized model's response and the expected output.

In [11]:
result["optimized_model_error"]

0.3503949046134949

`optimized_scaling_factors` is the scaling factor for each training iteration that was applied to the model to produce the optimized model.

In [12]:
result["optimized_scaling_factors"]

[-0.08874866916124041,
 -0.17300100314109593,
 -0.1752716729481579,
 -0.19940504499949116]

`number_of_inferences` is the number of inferences that were performed to produce the optimized model.

In [13]:
result["number_of_inferences"]

60

### Inference the non-optimized and optimized models

In [14]:
prompt_name = "prompt_00.json"
for i in range(10):
    !wget -O prompt_0{i}.json https://raw.githubusercontent.com/Authentrics-ai/demos/refs/heads/main/ZeroTrain_Optimizer_And_Maintenance/MedicalChatbot/prompt_0{i}.json
!wget -O expected-output.txt https://raw.githubusercontent.com/Authentrics-ai/demos/refs/heads/main/ZeroTrain_Optimizer_And_Maintenance/MedicalChatbot/expected-output.txt
prompt_path = Path(prompt_name)

--2026-01-23 16:37:25--  https://raw.githubusercontent.com/Authentrics-ai/demos/refs/heads/release/AA-1128-1221/ZeroTrain_Optimizer_And_Maintenance/MedicalChatbot/prompt_00.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 634 [text/plain]
Saving to: ‘prompt_00.json’

prompt_00.json      100%[===================>]     634  --.-KB/s    in 0s      

2026-01-23 16:37:25 (27.5 MB/s) - ‘prompt_00.json’ saved [634/634]



In [15]:
%%time
import json

stored_results = client.result.getAnalysisResults(project["id"])

optimized_model_path = f"{project_id}/results/{stored_results[-1]["requestId"]}.tar"
non_optimized_model_path = client.project.get_project_by_id(project_id)['fileList'][-1]['filePath']

non_optimized_response = client.dynamic.direct_inference(
    model_path=non_optimized_model_path,
    stimulus_path=prompt_path,
    format="HF_TEXT_GENERATION",
    inference_config={"max_new_tokens": 100},
)
optimized_response = client.dynamic.direct_inference(
    model_path=optimized_model_path,
    stimulus_path=prompt_path,
    format="HF_TEXT_GENERATION",
    inference_config={"max_new_tokens": 100},
)

non_optimized_output = non_optimized_response['result']["output"][0][0]["generated_text"][1]["content"]
optimized_output = optimized_response['result']["output"][0][0]["generated_text"][1]["content"]

with open(prompt_path, "r") as f:
    prompt = json.load(f)[0]["content"]

print(f"User prompt:\n\n\t{prompt}\n")
print(
    "Non-optimized model response:"
    f"\n\n\t{non_optimized_output}\n"
)
print(
    "Optimized model response:"
    f"\n\n\t{optimized_output}\n"
)

User prompt:

	Hello sir,  My son has sinusitis and also nasal polyps, we got to know this recently. He suffers from a nose block and breathing issues. So when we visited a PCP (primary care physician), he suggested using nasal spray. So my doubt is, how can nasal spray efficiently relieve symptoms of sinusitis and nasal polyps, and what mechanisms make it useful for people suffering from these conditions? Could you also elaborate on how they work to reduce inflammation and congestion, and any possible negative effects or things to watch out for when using nasal sprays to treat sinusitis and nasal polyps?   .

Non-optimized model response:

	I read your query and can understand your concern.  Nasal sprays work by releasing medications directly into the nasal passages to treat sinusitis and nasal polyps. Sinusitis causes inflammation and congestion in the nasal passages, leading to nasal blockages. Nasal sprays help by reducing inflammation and congestion, allowing the nasal passages to

## Example of error function used to optimize the model

In [16]:
def compute_error_score(ss_model: SentenceTransformer, model_outputs: list[str], expected_outputs: list[str]):
    if len(model_outputs) != len(expected_outputs):
        raise ValueError("model_output and expected_output must have the same length")

    generated_embeddings = ss_model.encode(model_outputs)
    expected_embeddings = ss_model.encode(expected_outputs)

    score = ss_model.similarity_pairwise(
        generated_embeddings,
        expected_embeddings,
    )

    return 1.0 - score

### Load prompts and expected outputs

In [17]:
import json

with open("expected_output.txt", "r") as f:
    expected_outputs = f.readlines()

prompts = []
for i in range(10):
    with open(f"prompt_{i:02d}.json", "r") as f:
        prompts.append(json.load(f)[0]["content"])

### Inference the models to get the model outputs

In [18]:
%%time
optimized_response = client.dynamic.batch_direct_inference(
    model_path=optimized_model_path,
    stimulus_paths=[
        f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)
    ],
    format="HF_TEXT_GENERATION",
    batch_size=10,
    inference_config={"max_new_tokens": 500},
)
non_optimized_response = client.dynamic.batch_direct_inference(
    model_path=non_optimized_model_path,
    stimulus_paths=[
        f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)
    ],
    format="HF_TEXT_GENERATION",
    batch_size=10,
    inference_config={"max_new_tokens": 500},
)

non_optimized_model_outputs = [response[0]["generated_text"][1]["content"] for response in non_optimized_response["result"]["output"]]
optimized_model_outputs = [response[0]["generated_text"][1]["content"] for response in optimized_response["result"]["output"]]

CPU times: user 7.19 ms, sys: 4.18 ms, total: 11.4 ms
Wall time: 33.3 s


### Scored responses with the Sentence Transformer we use internally

In [19]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

non_optimized_model_scores = compute_error_score(model, non_optimized_model_outputs, expected_outputs)
optimized_model_scores = compute_error_score(model, optimized_model_outputs, expected_outputs)

pd.DataFrame({
    "prompt": prompts,
    "non_optimized_model_score": non_optimized_model_scores,
    "optimized_model_score": optimized_model_scores,
})


,prompt,non_optimized_model_score,optimized_model_score
0,"Hello sir, My son has sinusitis and also nasa...",0.499,0.505
1,I read in the news that coronavirus is again s...,1.026,1.016
2,I am a 30-year-old female working as a traffic...,0.971,0.854
3,I have a question about my father. He is 75 ye...,0.905,0.842
4,"My child is 10 years old. As a parent, I want ...",1.088,1.010
5,"The fasting blood sugar of my mother, aged 55 ...",0.848,0.868
6,I had a sudden hearing loss twice and because ...,0.975,0.890
7,I am a 23-year-old female. I would like to kno...,1.066,1.056
8,"I am a 22-year-old female weighing 90 pounds, ...",0.916,1.002
9,Last night I looked directly at a 55 watt germ...,0.704,0.831


In [20]:
print(f'Average error on non-optimized model: {non_optimized_model_scores.mean().item():.3f}')
print(f'Average error on optimized model: {optimized_model_scores.mean().item():.3f}')
print(f"Overall loss improvement: {(non_optimized_model_scores - optimized_model_scores).mean():.3f}")
print(f"Overall loss percentage improvement: {(non_optimized_model_scores - optimized_model_scores).mean() / non_optimized_model_scores.mean() * 100:.2f}%")

Average error on non-optimized model: 0.900
Average error on optimized model: 0.887
Overall loss improvement: 0.012
Overall loss percentage improvement: 1.37%


## Analyze the ZTOM model

Upload the optimized model to the project


In [21]:
project = client.checkpoint.add_external_checkpoint(
    project_id,
    optimized_model_path,
    authrx.FileType.HF_TEXT,
    file_name="optimized_model.tar",
    tag="optimized",
)
checkpoint_id = project["fileList"][-1]["id"]

Wait for analytics to be ready


In [22]:
analytics_ready = wait_for_analytics(project_id, min_wait_time=10, max_wait_time=300)

Run static analysis on the optimized model

In [23]:
%%time
print("Running static analysis...")
with loading_spinner("Running static analysis..."):
    response = client.static.static_analysis(
        project_id=project_id,
        checkpoint_id=checkpoint_id,
    )
print("✅ Static analysis complete")

Running static analysis...


✅ Static analysis complete
CPU times: user 86.9 ms, sys: 49.9 ms, total: 137 ms
Wall time: 3.61 s


In [24]:
display_static_analysis_results(response["result"])


Weight summary score: 0.00017236183339264244


,name,min,mean,std,max
0,base_model.model.model.layers.0.mlp.down_proj....,-4.311e-05,-2.528e-07,1.185e-05,3.292e-05
1,base_model.model.model.layers.0.mlp.down_proj....,-5.955e-05,2.675e-08,2.042e-05,7.518e-05
2,base_model.model.model.layers.0.mlp.gate_proj....,-8.555e-05,3.187e-07,2.960e-05,9.618e-05
3,base_model.model.model.layers.0.mlp.gate_proj....,-3.882e-05,-5.321e-07,9.960e-06,2.822e-05
4,base_model.model.model.layers.0.mlp.up_proj.lo...,-1.058e-04,-2.365e-06,3.150e-05,9.240e-05
5,base_model.model.model.layers.0.mlp.up_proj.lo...,-4.573e-05,-5.761e-07,1.413e-05,3.586e-05
6,base_model.model.model.layers.0.self_attn.k_pr...,-5.869e-05,-1.901e-07,2.266e-05,6.598e-05
7,base_model.model.model.layers.0.self_attn.k_pr...,-7.913e-05,-9.753e-08,2.369e-05,7.722e-05
8,base_model.model.model.layers.0.self_attn.o_pr...,-1.384e-04,3.138e-06,3.534e-05,1.396e-04
9,base_model.model.model.layers.0.self_attn.o_pr...,-6.092e-05,-7.223e-08,2.244e-05,6.235e-05


## Summary

This notebook demonstrates how the Zero-Train Optimization & Maintenance (ZTOM) improves the Medical Chatbot's response quality without requiring any retraining. Using semantic similarity to guide optimization, ZTOM reduced the model's error rate by approximately 1.37%, demonstrating measurable improvements in response quality with minimal data requirements.
  
## ZTOM Result Outputs

  | Metric                       | Value                          |
  |------------------------------|-------------------------------:|
  | Non-Optimized Model Error    | 0.387                          |
  | Optimized Model Error        | 0.350                          |
  | Improvement                  | ~1.37%                         |
  | Scaling Factors              | [-0.089,-0.173,-0.175,-0.199]  |
  | Number of Inferences         | 60                             |